# Trinity THLP MC Benchmark

**Brain Zone:** Hippocampus / CA3-CA1  
**Dataset:** [`playra/trinity-cognitive-probes-thlp-mc`](https://www.kaggle.com/datasets/playra/trinity-cognitive-probes-thlp-mc)  
**Framework:** `kaggle-benchmarks` SDK  
**Track:** Trinity Hippocampal Learning Probe


In [ ]:
!pip install protobuf==5.29.6 --quiet
!pip install -q kaggle-benchmarks kaggle
import kaggle_benchmarks as kbench
print(f"[THLP] Step 0: kbench v{kbench.__version__}")

In [ ]:
import os, re, glob
os.environ["RENDER_SUBRUNS"] = "False"
import kaggle_benchmarks as kbench
import kaggle
import pandas as pd
from dataclasses import dataclass
print("[THLP] Step 1: Imports OK")

In [ ]:
print("[THLP] Step 2: Downloading playra/trinity-cognitive-probes-thlp-mc")
os.makedirs('/kaggle/working/data', exist_ok=True)
kaggle.api.dataset_download_files('playra/trinity-cognitive-probes-thlp-mc',
    path='/kaggle/working/data', unzip=True)

csv_files = glob.glob('/kaggle/working/data/**/*.csv', recursive=True)
print(f"[THLP] Available CSVs: {csv_files}")

CSV_PATH = next((f for f in csv_files if 'thlp_mc' in f.lower()), None)
if CSV_PATH is None:
    raise FileNotFoundError(f'[THLP] thlp_mc.csv not found in {csv_files}')
print(f"[THLP] Step 2: Using {CSV_PATH}")

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"[THLP] Loaded: {len(df)} rows, cols={list(df.columns)}")

if 'question_type' in df.columns:
    df = df[df['question_type'] == 'mc'].copy()
print(f"[THLP] MC rows: {len(df)}")

valid_mask = df['answer'].astype(str).str.strip().str.upper().str.match(r'^[A-D]$')
print(f"[THLP] Answer validity: {valid_mask.mean():.1%}")
assert valid_mask.mean() > 0.95, 'Answer validity below 95%'
assert df['choices'].isna().sum() == 0, 'Null choices found'

dist = df['answer'].value_counts(normalize=True).sort_index()
print(f"[THLP] Answer dist: {dict(dist.round(3))}")

eval_df = df[['question', 'choices', 'answer']].rename(
    columns={'answer': 'expected_answer'}).reset_index(drop=True)
print(f"[THLP] Step 3: eval_df ready, {len(eval_df)} questions")

In [ ]:
@dataclass
class MCAnswer:
    answer: str
print("[THLP] Step 4: MCAnswer schema defined")

In [ ]:
@kbench.task(name='thlp_single_mc', store_task=False)
def thlp_single_mc(llm, question: str, choices: str,
                   expected_answer: str) -> bool:
    prompt = f"""{question}

{choices}

Answer with ONLY ONE letter (A, B, C, or D).
Do not explain. Return exactly one character."""
    resp = llm.prompt(prompt, schema=MCAnswer)
    got = resp.answer.strip().upper()[:1]
    exp = str(expected_answer).strip().upper()[:1]
    return got == exp

print("[THLP] Step 5: inner task defined")

In [ ]:
@kbench.task(name='thlp_mc_benchmark',
             description='Hippocampal Learning Probe')
def thlp_mc_benchmark(llm) -> float:
    with kbench.client.enable_cache():
        runs = thlp_single_mc.evaluate(
            llm=[llm],
            evaluation_data=eval_df,
            n_jobs=2, timeout=180,
            max_attempts=1, remove_run_files=True)
    df_res = runs.as_dataframe()
    valid = df_res[df_res['result'].notna()]
    correct = int(valid['result'].sum())
    total = len(eval_df)
    acc = float(valid['result'].mean()) if len(valid) > 0 else 0.0
    print(f'[THLP] Valid: {len(valid)}/{total}, Correct: {correct}, Acc: {acc:.2%}')
    kbench.assertions.assert_true(True,
        expectation=f'THLP MC accuracy: {acc:.2%} ({correct}/{total})')
    return acc

run = thlp_mc_benchmark.run(kbench.llm)
print(f'[THLP] MC Accuracy: {run.result:.1%}')
%choose thlp_mc_benchmark